# Bundesliga 2023/2024: Tracking Data Extraction

#### 1. Import Required Libraries

In [3]:
import duckdb
import pandas as pd
import numpy as np
from typing import List
import matplotlib.pyplot as plt

#### 2. Load and Explore the Dataset

In [2]:
# Connection to DuckDB
conn = duckdb.connect('Bundesliga_2023_2024.duckdb')

In [3]:
# Helper to inspect DB schema
def show_database_schema(conn):
    """Zeigt das komplette Datenbankschema"""
    tables = conn.execute("SHOW TABLES").fetchall()
    
    for table_name in tables:
        table = table_name[0]
        print(f"\nTabelle: {table}")
        print("-" * 60)
        schema = conn.execute(f"DESCRIBE {table}").df()
        print(schema)

# Usage:
show_database_schema(conn)


Tabelle: match_information
------------------------------------------------------------
            column_name column_type null   key default extra
0                    id     VARCHAR  YES  None    None  None
1           match_title     VARCHAR  YES  None    None  None
2              matchday     INTEGER  YES  None    None  None
3  planned_kickoff_time   TIMESTAMP  YES  None    None  None
4          home_team_id     VARCHAR  YES  None    None  None
5          away_team_id     VARCHAR  YES  None    None  None
6                result     VARCHAR  YES  None    None  None

Tabelle: players
------------------------------------------------------------
     column_name column_type null   key default extra
0             id     VARCHAR  YES  None    None  None
1           name     VARCHAR  YES  None    None  None
2     first_name     VARCHAR  YES  None    None  None
3      last_name     VARCHAR  YES  None    None  None
4     short_name     VARCHAR  YES  None    None  None
5      birthdate    

In [4]:
# Match list query (unique matches + metadata)
query = """
    SELECT
        distinct tr.match_id, mi.match_title, mi.matchday, mi.result
    FROM
        tracking_raw_observed AS tr
    LEFT JOIN  match_information mi
        ON mi.id = tr.match_id
    ORDER BY matchday;
"""

# Execute
match_df = conn.execute(query).fetchdf()

# Display
match_df

,match_id,match_title,matchday,result
0,DFL-MAT-J03YDY,Eintracht Frankfurt:SV Darmstadt 98,1,1:0
1,DFL-MAT-J03YE1,FC Augsburg:Borussia Mönchengladbach,1,4:4
2,DFL-MAT-J03YDU,SV Werder Bremen:FC Bayern München,1,0:4
3,DFL-MAT-J03YDV,Borussia Dortmund:1. FC Köln,1,1:0
4,DFL-MAT-J03YE2,VfB Stuttgart:VfL Bochum 1848,1,5:0
...,...,...,...,...
301,DFL-MAT-J03YM3,Borussia Dortmund:SV Darmstadt 98,34,4:0
302,DFL-MAT-J03YM9,SV Werder Bremen:VfL Bochum 1848,34,4:1
303,DFL-MAT-J03YM6,Eintracht Frankfurt:RB Leipzig,34,2:2
304,DFL-MAT-J03YMA,VfB Stuttgart:Borussia Mönchengladbach,34,4:0


In [5]:
# Selected match_id
match_id = "DFL-MAT-J03YLO"

In [6]:
# Team metadata for selected match
query = """
    SELECT
        mi.id,
        mi.match_title,
        home.three_letter_code AS home_team,
        away.three_letter_code AS away_team,
        mi.matchday
    FROM
        match_information AS mi
    LEFT JOIN teams AS home
        ON mi.home_team_id = home.id
    LEFT JOIN teams AS away
        ON mi.away_team_id = away.id
    ORDER BY mi.matchday;
"""

# Execute
teams = conn.execute(query).fetchdf()

# Display
teams
teams[teams['id'] == match_id]



,id,match_title,home_team,away_team,matchday
280,DFL-MAT-J03YLO,VfL Wolfsburg:SV Darmstadt 98,WOB,SVD,32


#### 3. Detailed Positional Data Extraction for Selected Match

In [9]:
# Full positional tracking for the selected match (includes player and team context)
query = f"""
    SELECT 
    tr.match_id,
    m.match_title,
    t.three_letter_code AS team_code,
    home.three_letter_code AS home_team_code,
    away.three_letter_code AS away_team_code,    
    p.short_name AS player_name,
    tr.frame,
    tr.game_section,
    tr.x_position,
    tr.y_position,
    tr.z_position,
    tr.speed,
    tr.ball_status,
    tr.ball_possession
    FROM tracking_raw_observed tr
    LEFT JOIN match_information m 
        ON tr.match_id = m.id
    LEFT JOIN teams t 
        ON tr.team_id = t.id
    LEFT JOIN teams AS home
        ON m.home_team_id = home.id
    LEFT JOIN teams AS away
        ON m.away_team_id = away.id        
    LEFT JOIN players p 
        ON tr.player_id = p.id
    WHERE match_id = '{match_id}'
    ORDER BY tr.frame, team_code, x_position;
"""

# Execute
pos = conn.execute(query).fetchdf()

#restart frame_column from 1
pos['frame'] = pos['frame'] - pos['frame'].min() + 1

# Display
pos.head()

,match_id,match_title,team_code,home_team_code,away_team_code,player_name,frame,game_section,x_position,y_position,z_position,speed,ball_status,ball_possession
0,DFL-MAT-J03YLO,VfL Wolfsburg:SV Darmstadt 98,SVD,WOB,SVD,A. Brunst,1,firstHalf,-43.860001,0.11,NaN,0.0,<NA>,None
1,DFL-MAT-J03YLO,VfL Wolfsburg:SV Darmstadt 98,SVD,WOB,SVD,C. Klarer,1,firstHalf,-16.719999,3.28,NaN,0.0,<NA>,None
2,DFL-MAT-J03YLO,VfL Wolfsburg:SV Darmstadt 98,SVD,WOB,SVD,C. Zimmermann,1,firstHalf,-16.410000,14.04,NaN,0.0,<NA>,None
3,DFL-MAT-J03YLO,VfL Wolfsburg:SV Darmstadt 98,SVD,WOB,SVD,M. Maglica,1,firstHalf,-16.370001,-4.52,NaN,0.0,<NA>,None
4,DFL-MAT-J03YLO,VfL Wolfsburg:SV Darmstadt 98,SVD,WOB,SVD,E. Karic,1,firstHalf,-12.430000,22.08,NaN,0.0,<NA>,None


In [10]:
# Close DB connection
conn.close()

In [11]:
# Save tracking data (CSV)
pos.to_csv(f'tracking_data_{match_id}.csv', index=False)